# 1. Components

Components are the molecules that a structure is built from: lipids, surfactants, ions and solvents.

In this notebook you will:

1. see the assets in the Shapespyer library
2. load a component from the library and look at it
3. make a component from a file of your own
4. make a component from a SMILES string
5. save your components to the library, so every notebook can load them

Run the setup cell first. On Colab it installs Shapespyer, which takes about a minute. Locally it only reads `shapespyer.yaml`, which says where Shapespyer keeps its files. Then run the helpers cell under it: it only draws molecules, downloads an example file and checks the library, and isn't part of the Shapespyer workflow.

In [ ]:
# @title Setup: on Colab, install Shapespyer (about 1 min) { display-mode: "form" }
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    %pip install -q "shapespyer==0.3.0"

    # nglview draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()
    # and without its frame player: one-frame views don't use it, and its jslink opens Colab's widget panel
    import nglview
    nglview.NGLWidget._create_player = lambda self: None

    # the library, Temp and Projects go to /content, to browse them in the Files panel
    from shapes.project import Config
    config = Config(CONFIG_FILE)
    config.app_path = "/content/shapespyer"
    config.save()
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Shapespyer files:", config.app_path)

In [ ]:
# @title Helpers (not part of the Shapespyer workflow) { display-mode: "form" }
import urllib.request

import mdtraj
import nglview

from shapes.project import Library

EXAMPLE_FILES_URL = "https://gitlab.com/simnavi/shapespyer/-/raw/v0.3.0/assets/components"


def show_structure(component):
    """Draw a component, as its .show() does; Shapespyer 0.3.0's .show() fails on Colab."""
    view = nglview.show_structure_file(str(component.structure_file_path))
    view.clear_representations()
    view.add_representation("ball+stick", selection=component.name, radius=0.25)
    return view


def download_example_file(path, folder):
    """Download a file of the Shapespyer repository into a folder, to stand in for a file of your own."""
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / path.rsplit("/", 1)[-1]
    urllib.request.urlretrieve(f"{EXAMPLE_FILES_URL}/{path}", target)
    return target


def show_component(component):
    """Draw only the residues named like the component, for a file that holds other molecules too."""
    frame = mdtraj.load(str(component.structure_file_path))
    atoms = frame.top.select(f"resname {component.name}")
    print(f"{component.name}: {len(atoms)} of the {frame.n_atoms} atoms in {component.structure_file_path.name}")
    return nglview.show_mdtraj(frame.atom_slice(atoms))


def show_library_entry(asset_type, name):
    """Print whether you saved a library entry yourself, and its files."""
    record = Library().find(asset_type, name=name)
    print(f"{name} saved by you:", "ser_data" in record)
    for path in record.get("paths", []):
        print(" ", path)

In [ ]:
import shutil

from shapes.basics.utils import LogConfiguration
from shapes.project import Library, Surfactant

LogConfiguration("WARNING")

library = Library()
# the downloaded example file goes to a work folder, which the clean up deletes
WORK = config.app_path / "work"

### The library

The first `Library()` of a session reads the remote index of assets and downloads their files into `LocalAssets/` in the application directory. On Colab that is `/content/shapespyer/LocalAssets`: open the Files panel on the left to see it.

These are the assets ready to use, by type:

In [ ]:
library.available_assets

### Load a component and look at it

`show_structure()`, from the helpers cell, draws the molecule, as the component's `.show()` does. Shapespyer 0.3.0's `.show()` fails on Colab. It must be the last line of the cell.

In [ ]:
sds = Surfactant.load("SDS")
show_structure(sds)

### A component from your own file

Here a helper downloads a `.gro` file from the Shapespyer repository, as if it were your own. The name of the component must match the residue name in the file. This file also holds an SDS molecule next to the CTA, and `show_structure()` would draw both, so `show_component()` draws only the residues named CTA.

On Colab you can also upload your own file: drag it into the Files panel, or run `from google.colab import files; files.upload()`.

In [ ]:
my_file = download_example_file("surfactants/CTA/CTA.gro", WORK)

cta = Surfactant(name="CTA", paths=[my_file])
show_component(cta)

### A component from a SMILES string

Here we make SDS from its SMILES string. Hydrogens are added, and the atoms follow the SMILES order. We write the tail first, so the "bone" of the molecule runs from the tail end (`mint` is 0) to the head (`mext` is the last atom), as a micelle needs.

The name becomes the residue name in the file, at most 5 characters. We call it `SDSsm`, so it doesn't clash with the library's SDS. A structure finds its molecules by this residue name, so a component can't be renamed later. The component gets a structure file only, with no topology. So it can be built into structures, but not equilibrated.

In [ ]:
sds_from_smiles = Surfactant.from_smiles(
    "SDSsm",
    "CCCCCCCCCCCCOS(=O)(=O)[O-]",
    box=(3.0, 3.0, 3.0),  # in nm
    formula="C12H25SO4",
    align_z=True,
)
print(sds_from_smiles.number_of_atoms, sds_from_smiles.mint, sds_from_smiles.mext)
show_structure(sds_from_smiles)

### Save your components to the library

`.save()` copies the files of a component into the library, `LocalAssets/<type>/<name>/`, with a `ser_data.yaml` record of the whole component. From then on, `Surfactant.load(name)` finds it: in this notebook, in the other notebooks and in later sessions.

We save our two components:

- `SDSsm` has a new name, so it is simply added.
- `CTA` has the name of a library asset, so saving it replaces that asset in your local library.

If you run this notebook a second time without the clean up at the end, use `overwrite=True` here too.

In [ ]:
sds_from_smiles.save()
library.available_assets["Surfactant"]

The name `CTA` is taken by the library's CTA, so a plain `.save()` refuses:

In [ ]:
show_library_entry("Surfactant", "CTA")
try:
    cta.save()
except ValueError as err:
    print(err)

`overwrite=True` replaces it. The library looks at your local entries first, and `sync()` leaves the entries you saved alone, so every notebook now loads your CTA. Here your file is the same as the library's; with a CTA of your own, such as one with other coordinates, yours would be used.

In [ ]:
cta.save(overwrite=True)
show_library_entry("Surfactant", "CTA")

Load a saved component back from the library, as any other notebook would. It works on the copies in the library, not on the files it was made from:

In [ ]:
loaded = Surfactant.load("SDSsm")
print(loaded.paths)
show_structure(loaded)

### Clean up (optional)

This deletes the work folder with the downloaded `CTA.gro`, and your two library entries. Without this, every later notebook and session would load your CTA instead of the library's. `force=True` is needed, because `loaded` still works on the files of its entry. The library's CTA comes back with the next lookup, which downloads it again.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
library.delete_entry("Surfactant", "SDSsm", force=True)
library.delete_entry("Surfactant", "CTA", force=True)

show_library_entry("Surfactant", "CTA")
print(WORK.exists(), library.available_assets["Surfactant"])